## Conexão dos Notebooks

<a href="https://colab.research.google.com/github/Yuoneeh/Projeto_Redes_EstDados/blob/main/notebooks/isaque/Tarefa4_Ajuste_da_Arvore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
from pathlib import Path

# Detecta se está no Google Colab ou em ambiente local
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_PROJETO = Path("/content/drive/MyDrive/Projeto interdisciplinar")
except (ImportError, ModuleNotFoundError):
    RAIZ_PROJETO = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]

# Configura e cria os caminhos absolutos
PASTA_DATA = RAIZ_PROJETO / "data"
PROC = PASTA_DATA / "processed"
INTERIM = PASTA_DATA / "interim"
PASTA_RAW = PASTA_DATA / "raw"
PASTA_SRC = RAIZ_PROJETO / "src"

# CRIAÇÃO DA PASTA MODELS EXIGIDA
PASTA_MODELS = RAIZ_PROJETO / "models"

for p in (INTERIM, PROC, PASTA_RAW, PASTA_SRC, PASTA_MODELS):
    p.mkdir(parents=True, exist_ok=True)

## Ajuste da Árvore


In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

SEED = 42
CLASSES = ["OK", "RISCO", "FALHA"]

df = pd.read_csv(PROC / "dataset_final_treino.csv")
FEATURES = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pacotes",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]

# Nas Tarefas 3, 4 e 5:
X = df[FEATURES].fillna({"z_robusto": 0.0, "aumento_pct": 0.0, "jitter_relativo": 1.0})
y = df["rotulo"]  # <-- Previsão antecipada de 12 minutos

# 1. Definição das máscaras para as três fases temporais
treino_mask = df["bloco"] == "treino"
val_mask = df["bloco"] == "validacao"
teste_mask = df["bloco"] == "teste"

# 2. Separação explícita de Treino, Validação e Teste
X_treino, y_treino = X[treino_mask], y[treino_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_teste, y_teste = X[teste_mask], y[teste_mask]

print(f"Instâncias de Treino:    {len(X_treino)} ({len(X_treino)/len(X):.1%})")
print(f"Instâncias de Validação: {len(X_val)} ({len(X_val)/len(X):.1%})")
print(f"Instâncias de Teste:     {len(X_teste)} ({len(X_teste)/len(X):.1%})")

# Reproduzir Árvore da Tarefa 3 para tabela comparativa
arvore_t3 = DecisionTreeClassifier(criterion="gini", max_depth=3, min_samples_leaf=50, random_state=SEED)
arvore_t3.fit(X_treino, y_treino)
pred_t3 = arvore_t3.predict(X_val)

# Busca na grade olhando SOMENTE a validação (sem tocar no teste!)
candidatos = []
for crit in ["gini", "entropy"]:
    for depth in [4, 5, 6, 8, 10]:
        for leaf in [5, 10, 20, 50]:
            for peso in [None, "balanced"]:
                clf = DecisionTreeClassifier(
                    criterion=crit, max_depth=depth, min_samples_leaf=leaf,
                    class_weight=peso, random_state=SEED
                )
                clf.fit(X_treino, y_treino)
                p_val = clf.predict(X_val)
                candidatos.append({
                    "criterion": crit, "max_depth": depth, "min_samples_leaf": leaf, "class_weight": peso,
                    "f1_macro": f1_score(y_val, p_val, average="macro", labels=CLASSES, zero_division=0),
                    "recall_falha": recall_score(y_val, p_val, labels=["FALHA"], average="macro", zero_division=0),
                    "recall_risco": recall_score(y_val, p_val, labels=["RISCO"], average="macro", zero_division=0),
                    "modelo": clf
                })

ranking = pd.DataFrame(candidatos).sort_values("f1_macro", ascending=False).reset_index(drop=True)
melhor = ranking.iloc[0]
arvore_t4 = melhor["modelo"]
pred_t4 = arvore_t4.predict(X_val)

print("Melhores parâmetros na validação:", {k: melhor[k] for k in ["criterion", "max_depth", "min_samples_leaf", "class_weight"]})
print(f"Profundidade obtida: {arvore_t4.get_depth()} | Folhas: {arvore_t4.get_n_leaves()}")

# Tabela obrigatória da Seção 3 da Tarefa 4 (Tarefa 3 -> Tarefa 4)
comparacao = pd.DataFrame({
    "F1 macro": [
        f1_score(y_val, pred_t3, average="macro", labels=CLASSES, zero_division=0),
        melhor["f1_macro"]
    ],
    "Recall de FALHA": [
        recall_score(y_val, pred_t3, labels=["FALHA"], average="macro", zero_division=0),
        melhor["recall_falha"]
    ],
    "Recall de RISCO": [
        recall_score(y_val, pred_t3, labels=["RISCO"], average="macro", zero_division=0),
        melhor["recall_risco"]
    ]
}, index=["Árvore da Tarefa 3", "Árvore desta tarefa (Tarefa 4)"]).round(4)

display(comparacao)
print("\nMatriz 3x3 da Árvore Ajustada (Validação):")
print(pd.DataFrame(confusion_matrix(y_val, pred_t4, labels=CLASSES),
                   index=[f"Real_{c}" for c in CLASSES], columns=[f"Pred_{c}" for c in CLASSES]))
print("\nNovas regras em português:")
print(export_text(arvore_t4, feature_names=FEATURES, max_depth=4))

# Salvar a árvore ajustada para entrar na Tarefa 5
joblib.dump(arvore_t4, PROC / "arvore_ajustada_tarefa4.joblib")